# Force-Resolution Correction — Multi-Epoch Validation

Evaluates the **multi-epoch** CNN+MLP pipeline trained with `multisim_cnn.yaml` + `multisim_mlp.yaml`.

**Data**: `generate_data_single.py --mode multisim`  
n_part=128, mesh_lr=128, mesh_hr=256, L=256 Mpc/h, 6 sims, 10 snaps (a=0.1–1.0)

| Section | Question |
|---------|----------|
| 0 | Config & checkpoint loading |
| 1 | Force pair sanity — distribution, signal strength |
| 2 | Sub-region split: train vs test |
| 3 | Model inference at a single snapshot |
| 4 | **Temporal generalisation** — R vs scale factor a |
| 5 | **Cross-simulation generalisation** — held-out sim |
| 6 | Spatial error maps |
| 7 | **On-the-fly fine-tuning** — skeleton for first tests |

In [ ]:
%load_ext autoreload
%autoreload 2

import os
os.environ["JAX_ENABLE_X64"] = "0"

import yaml, pickle
from pathlib import Path
from types import SimpleNamespace
from functools import partial

import numpy as np
import jax
import jax.numpy as jnp
import optax
import haiku as hk
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from scipy.stats import pearsonr

import sys
REPO_ROOT = Path("../").resolve()
sys.path.insert(0, str(REPO_ROOT / "pm2nbody"))

from train_lag_force          import compute_force_pair, snapshot_features
from train_subregion_forceres import load_single_snapshot, make_patch_split
from train_lag_massres        import (
    _load_cnn_massres_checkpoint,
    compute_cnn_massres_correction,
)
from jaxpm.lagrangian import get_axis_neighbor_indices, make_lagrangian_corrector

print("imports OK  |  JAX:", jax.devices())

## Section 0 — Config & checkpoint loading

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  CONFIG — switch between multisim (new) and subregion (old)
# ═══════════════════════════════════════════════════════════════════
USE_MULTISIM = True   # True → multisim_mlp.yaml; False → subregion_forceres.yaml

if USE_MULTISIM:
    MLP_CFG_PATH  = REPO_ROOT / "configs/multisim_mlp.yaml"
    CNN_CFG_PATH  = REPO_ROOT / "configs/multisim_cnn.yaml"
    MLP_CKPT_DIR  = REPO_ROOT / "runs/multisim_mlp"
    CNN_CKPT_DIR  = REPO_ROOT / "runs/multisim_cnn"
else:
    MLP_CFG_PATH  = REPO_ROOT / "configs/subregion_forceres.yaml"
    CNN_CFG_PATH  = None
    MLP_CKPT_DIR  = REPO_ROOT / "runs/subregion_forceres"
    CNN_CKPT_DIR  = None

with open(MLP_CFG_PATH) as f:
    cfg = yaml.safe_load(f)

data_cfg  = SimpleNamespace(**cfg["data"])
model_cfg = SimpleNamespace(**cfg["model"])
train_cfg = SimpleNamespace(**cfg["training"])

DATA_DIR   = Path(data_cfg.data_dir)
N_PART     = int(data_cfg.n_part)
MESH_LR    = int(data_cfg.mesh_lr)
MESH_HR    = int(data_cfg.mesh_hr)
BOX_SIZE   = float(data_cfg.box_size)
SCALE_TO_LR = float(MESH_LR) / float(N_PART)

# Training snapshots
_s = getattr(data_cfg, "snaps_train", None) or getattr(data_cfg, "snap_train", 5)
SNAPS_TRAIN = [int(_s)] if isinstance(_s, (int, float)) else [int(x) for x in _s]
_sv = getattr(data_cfg, "snaps_val", [2, 5, 8])
SNAPS_VAL   = [int(x) for x in _sv]

# Sims (multi-sim or single-sim)
SIM_IDS_TRAIN = list(getattr(data_cfg, "sim_ids_train", [getattr(data_cfg, "sim_id_train", 0)]))
SIM_IDS_VAL   = list(getattr(data_cfg, "sim_ids_val",   [getattr(data_cfg, "sim_id_val",   1)]))

# Sub-region for on-the-fly fine-tuning (default: 12.5% of particles)
TRAIN_PATCH_N = int(getattr(train_cfg, "train_patch_n", N_PART // 2))
PATCH_SEED    = 42

USE_STRAIN = bool(model_cfg.use_strain)
USE_INVS   = bool(model_cfg.use_invariants)
USE_VEL    = bool(model_cfg.use_velocity)

train_frac = (TRAIN_PATCH_N / N_PART) ** 3
print(f"n_part={N_PART}  mesh_lr={MESH_LR}  mesh_hr={MESH_HR}  box={BOX_SIZE} Mpc/h")
print(f"snaps_train={SNAPS_TRAIN}  snaps_val={SNAPS_VAL}")
print(f"sim_ids_train={SIM_IDS_TRAIN}  sim_ids_val={SIM_IDS_VAL}")
print(f"patch={TRAIN_PATCH_N}³ = {TRAIN_PATCH_N**3:,} / {N_PART**3:,} particles ({train_frac:.1%})")

In [ ]:
# ── Load MLP checkpoint ───────────────────────────────────────────────────────
def load_latest_checkpoint(ckpt_dir):
    for rd in sorted(Path(ckpt_dir).glob("*/"), key=lambda p: p.stat().st_mtime, reverse=True):
        for fname in ["best_params.pkl", "final_params.pkl", "checkpoint.pkl"]:
            pkl = rd / fname
            if pkl.exists():
                with open(pkl, "rb") as fh:
                    return hk.data_structures.to_immutable_dict(pickle.load(fh)), pkl
    return None, None

lag_model = make_lagrangian_corrector(
    hidden_dim=int(model_cfg.hidden_dim),
    n_layers=int(model_cfg.n_layers),
    output_dim=3,
)

MLP_PARAMS, mlp_ckpt = load_latest_checkpoint(MLP_CKPT_DIR)
if MLP_PARAMS is not None:
    n_mlp = sum(x.size for x in jax.tree_util.tree_leaves(MLP_PARAMS))
    print(f"MLP loaded: {mlp_ckpt}  ({n_mlp:,} params)")
else:
    print("⚠  No MLP checkpoint — run train_pretrain_forceres.py first")

# ── Load CNN checkpoint ───────────────────────────────────────────────────────
CNN_MODEL, CNN_PARAMS = None, None
IS_TWO_STAGE = False

cnn_ckpt_path = getattr(model_cfg, "cnn_checkpoint", None)
if cnn_ckpt_path is None and CNN_CKPT_DIR is not None:
    # Auto-find CNN checkpoint from runs/multisim_cnn
    _, auto_cnn = load_latest_checkpoint(CNN_CKPT_DIR)
    if auto_cnn is not None:
        cnn_ckpt_path = str(auto_cnn)

if cnn_ckpt_path is not None and Path(cnn_ckpt_path).exists():
    CNN_MODEL, CNN_PARAMS = _load_cnn_massres_checkpoint(cnn_ckpt_path)
    IS_TWO_STAGE = True
    n_cnn = sum(x.size for x in jax.tree_util.tree_leaves(CNN_PARAMS))
    print(f"CNN loaded: {cnn_ckpt_path}  ({n_cnn:,} params)")
else:
    print("CNN: not loaded (single-stage MLP only)")

STAGE_LABEL = "CNN + MLP" if IS_TWO_STAGE else "MLP only"
print(f"\nPipeline: {STAGE_LABEL}")

## Section 1 — Force pair sanity (single snapshot)

In [ ]:
# Load a representative snapshot for inspection
SNAP_INSPECT = SNAPS_TRAIN[len(SNAPS_TRAIN) // 2]   # middle training snap
SIM_INSPECT  = SIM_IDS_TRAIN[0]

neighbor_idx = get_axis_neighbor_indices(N_PART)
train_idx, test_idx, _ = make_patch_split(N_PART, TRAIN_PATCH_N, seed=PATCH_SEED)

pos, vel, a = load_single_snapshot(DATA_DIR, SIM_INSPECT, SNAP_INSPECT, N_PART, BOX_SIZE)
pos_lr = pos * SCALE_TO_LR

_fp = jax.jit(partial(compute_force_pair, mesh_lr=MESH_LR, mesh_hr=MESH_HR))
f_coarse, f_fine, delta_f = _fp(pos_lr)

f_coarse_np = np.asarray(jax.device_get(f_coarse))
f_fine_np   = np.asarray(jax.device_get(f_fine))
delta_f_np  = np.asarray(jax.device_get(delta_f))
pos_np      = np.asarray(jax.device_get(pos))
pos_lr_np   = np.asarray(jax.device_get(pos_lr))

df_mag = np.sqrt(np.sum(delta_f_np**2, axis=-1))
fc_mag = np.sqrt(np.sum(f_coarse_np**2, axis=-1))
ff_mag = np.sqrt(np.sum(f_fine_np**2,   axis=-1))

print(f"sim={SIM_INSPECT}  snap={SNAP_INSPECT}  a={a:.3f}")
print(f"|F_coarse| mean = {fc_mag.mean():.4e}")
print(f"|F_fine|   mean = {ff_mag.mean():.4e}")
print(f"|ΔF|       mean = {df_mag.mean():.4e}  ({df_mag.mean()/ff_mag.mean():.1%} of F_fine)")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
ss = np.random.default_rng(0).choice(len(df_mag), min(50_000, len(df_mag)), replace=False)

axes[0].hist(delta_f_np[:, 0], bins=200, alpha=0.7, density=True, color="C0")
axes[0].set_xlabel("ΔF_x [mesh units]"); axes[0].set_title("ΔF_x distribution")

axes[1].hexbin(ff_mag[ss], df_mag[ss], gridsize=70, norm=LogNorm(), mincnt=1, cmap="plasma")
axes[1].set_xlabel("|F_fine|"); axes[1].set_ylabel("|ΔF|")
axes[1].set_title(f"|ΔF| vs |F_fine|  a={a:.3f}")

r_ff, _ = pearsonr(fc_mag[ss], ff_mag[ss])
axes[2].hexbin(ff_mag[ss], fc_mag[ss], gridsize=70, norm=LogNorm(), mincnt=1, cmap="viridis")
axes[2].plot([0, ff_mag.max()], [0, ff_mag.max()], "r--", lw=1, label="identity")
axes[2].set_xlabel("|F_fine|"); axes[2].set_ylabel("|F_coarse|")
axes[2].set_title(f"R={r_ff:.4f}  (coarse vs fine)")
axes[2].legend()

plt.suptitle(f"Force pair  (mesh_lr={MESH_LR} vs mesh_hr={MESH_HR})  a={a:.3f}")
plt.tight_layout(); plt.show()

## Section 2 — Sub-region split (Lagrangian patch)

In [ ]:
q = np.stack(np.meshgrid(*[np.arange(N_PART)]*3, indexing='ij'), axis=-1).reshape(-1, 3)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ax = axes[0]
ax.scatter(q[test_idx,  0], q[test_idx,  1], s=0.3, c="steelblue", alpha=0.3, label="test")
ax.scatter(q[train_idx, 0], q[train_idx, 1], s=0.5, c="tomato",    alpha=0.8, label="train")
ax.set_title(f"Lagrangian split  (patch={TRAIN_PATCH_N}³, {train_frac:.1%})")
ax.set_xlabel("ix"); ax.set_ylabel("iy"); ax.legend(markerscale=6)

ax = axes[1]
for idx, label, col in [(test_idx, "test", "steelblue"), (train_idx, "train", "tomato")]:
    ax.scatter(pos_np[idx, 0], pos_np[idx, 1], s=0.3, c=col, alpha=0.4, label=label)
ax.set_title("Eulerian view (train=red, test=blue)")
ax.set_xlabel("x [n_part units]"); ax.legend(markerscale=6)

plt.tight_layout(); plt.show()

print(f"Train particles: {len(train_idx):,}  |  Test particles: {len(test_idx):,}")

## Section 3 — Model inference at a single snapshot

In [ ]:
def predict_all(params, cnn_model, cnn_params, pos, vel, a, mesh_lr, mesh_hr, scale_to_lr, use_vel):
    """Returns (total_pred, cnn_pred, mlp_pred, delta_f, f_coarse, f_fine) as numpy arrays."""
    pos_lr_j = pos * scale_to_lr
    f_c, f_f, df = jax.jit(partial(compute_force_pair, mesh_lr=mesh_lr, mesh_hr=mesh_hr))(pos_lr_j)

    feats, _ = snapshot_features(pos, neighbor_idx, N_PART, USE_STRAIN, USE_INVS)
    vel_feat  = vel if use_vel else jnp.zeros_like(vel)
    mlp_p     = jax.jit(params["lag_model"].apply if isinstance(params, dict) and "lag_model" in params
                        else lag_model.apply)(params, feats, vel_feat, jnp.array(float(a)))

    if cnn_model is not None:
        vel_lr = vel * scale_to_lr
        cnn_p  = jax.jit(lambda p, v, a_: compute_cnn_massres_correction(
            cnn_model, cnn_params, p, v, a_, mesh_lr
        ))(pos_lr_j, vel_lr, jnp.array(float(a)))
    else:
        cnn_p = jnp.zeros_like(mlp_p)

    total = np.asarray(jax.device_get(cnn_p)) + np.asarray(jax.device_get(mlp_p))
    return (total, np.asarray(jax.device_get(cnn_p)), np.asarray(jax.device_get(mlp_p)),
            np.asarray(jax.device_get(df)), np.asarray(jax.device_get(f_c)), np.asarray(jax.device_get(f_f)))


def region_metrics(pred, df_np, fc_np, ff_np, idx):
    mse = float(np.mean((pred[idx] - df_np[idx])**2))
    r_xyz = [pearsonr(pred[idx, c], df_np[idx, c])[0] for c in range(3)]
    base  = float(np.mean((fc_np[idx] - ff_np[idx])**2))
    corr  = float(np.mean((fc_np[idx] + pred[idx] - ff_np[idx])**2))
    return dict(r=float(np.mean(r_xyz)), r_xyz=r_xyz, mse_improv=(1 - corr / base) * 100)


if MLP_PARAMS is not None:
    total_np, cnn_np, mlp_np, df_np, fc_np, ff_np = predict_all(
        MLP_PARAMS, CNN_MODEL, CNN_PARAMS, pos, vel, a,
        MESH_LR, MESH_HR, SCALE_TO_LR, USE_VEL
    )

    print(f"\n{'Region':<8} | {'R_mean':>8} | {'MSE_improv%':>12} | {'R_x':>6} {'R_y':>6} {'R_z':>6}")
    print("─" * 58)
    for region, idx in [("train", train_idx), ("test", test_idx),
                        ("full",  np.arange(len(total_np)))]:
        m = region_metrics(total_np, df_np, fc_np, ff_np, idx)
        print(f"{region:<8} | {m['r']:>8.4f} | {m['mse_improv']:>11.1f}% | "
              f"{m['r_xyz'][0]:>6.3f} {m['r_xyz'][1]:>6.3f} {m['r_xyz'][2]:>6.3f}")

    gap = region_metrics(total_np, df_np, fc_np, ff_np, train_idx)["r"] - \
          region_metrics(total_np, df_np, fc_np, ff_np, test_idx)["r"]
    print(f"\nGeneralisation gap (train R − test R) = {gap:+.4f}")

    # Scatter plot
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    for row, (idx, label) in enumerate([(train_idx, "Train"), (test_idx, "Test")]):
        ss2 = np.random.default_rng(row).choice(len(idx), min(30_000, len(idx)), replace=False)
        i2  = idx[ss2]
        for ci, comp in enumerate(["x", "y", "z"]):
            t, p = df_np[i2, ci], total_np[i2, ci]
            lim  = max(abs(np.percentile(t, 1)), abs(np.percentile(t, 99))) * 1.15
            ax   = axes[row, ci]
            h = ax.hexbin(t, p, gridsize=60, cmap="Blues" if row==0 else "Greens",
                          norm=LogNorm(), mincnt=1, extent=[-lim, lim, -lim, lim])
            plt.colorbar(h, ax=ax)
            ax.plot([-lim, lim], [-lim, lim], "r--", lw=1)
            ax.set_title(f"{label} {comp}  R={pearsonr(t, p)[0]:.4f}")
    plt.suptitle(f"{STAGE_LABEL}  a={a:.3f}  sim={SIM_INSPECT}", fontsize=11)
    plt.tight_layout(); plt.show()
else:
    print("No MLP checkpoint loaded.")

## Section 4 — Temporal generalisation: R vs scale factor a

Evaluates R on **all 10 snapshots** for both the training sim and the held-out validation sim.
A model that generalises across time should show consistent R regardless of a.

In [ ]:
import pandas as pd

ALL_SNAPS = list(range(10))   # all 10 snapshots

def eval_all_snaps(sim_id, snap_list, params, cnn_model, cnn_params):
    rows = []
    _fp_jit = jax.jit(partial(compute_force_pair, mesh_lr=MESH_LR, mesh_hr=MESH_HR))
    for snap_id in snap_list:
        p, v, a_s = load_single_snapshot(DATA_DIR, sim_id, snap_id, N_PART, BOX_SIZE)
        p_lr = p * SCALE_TO_LR
        fc, ff, df = _fp_jit(p_lr)
        feats, _ = snapshot_features(p, neighbor_idx, N_PART, USE_STRAIN, USE_INVS)
        vf = v if USE_VEL else jnp.zeros_like(v)
        mlp_p = np.asarray(jax.device_get(
            jax.jit(lag_model.apply)(params, feats, vf, jnp.array(float(a_s)))
        ))
        if cnn_model is not None:
            v_lr = v * SCALE_TO_LR
            cnn_p = np.asarray(jax.device_get(jax.jit(lambda pp, vv, aa:
                compute_cnn_massres_correction(cnn_model, cnn_params, pp, vv, aa, MESH_LR)
            )(p_lr, v_lr, jnp.array(float(a_s)))))
        else:
            cnn_p = np.zeros_like(mlp_p)

        total = cnn_p + mlp_p
        df_np = np.asarray(jax.device_get(df))
        fc_np = np.asarray(jax.device_get(fc))
        ff_np = np.asarray(jax.device_get(ff))

        def r_m(idx): return float(np.mean([pearsonr(total[idx,c], df_np[idx,c])[0] for c in range(3)]))
        def mse_i(idx):
            base = float(np.mean((fc_np[idx] - ff_np[idx])**2))
            corr = float(np.mean((fc_np[idx] + total[idx] - ff_np[idx])**2))
            return (1 - corr / base) * 100

        trained_snap = snap_id in SNAPS_TRAIN
        rows.append(dict(
            sim=sim_id, snap=snap_id, a=float(a_s),
            trained_on=trained_snap,
            train_R=r_m(train_idx), test_R=r_m(test_idx),
            full_R=r_m(np.arange(len(total))),
            train_MSE=mse_i(train_idx), test_MSE=mse_i(test_idx),
        ))
        del p, v, p_lr, fc, ff, df, feats, vf, mlp_p, total
    return pd.DataFrame(rows)


if MLP_PARAMS is not None:
    print("Evaluating training sim across all snapshots…")
    df_train_sim = eval_all_snaps(SIM_IDS_TRAIN[0], ALL_SNAPS, MLP_PARAMS, CNN_MODEL, CNN_PARAMS)

    print("Evaluating held-out val sim across all snapshots…")
    df_val_sim   = eval_all_snaps(SIM_IDS_VAL[0],   ALL_SNAPS, MLP_PARAMS, CNN_MODEL, CNN_PARAMS)

    # ── Table ────────────────────────────────────────────────────────────────
    print("\n=== Training sim ===")
    print(df_train_sim[["snap","a","trained_on","train_R","test_R","test_MSE"]].to_string(index=False, float_format="{:.4f}".format))
    print("\n=== Validation sim (held-out) ===")
    print(df_val_sim[["snap","a","trained_on","train_R","test_R","test_MSE"]].to_string(index=False, float_format="{:.4f}".format))

In [ ]:
if MLP_PARAMS is not None:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for df_s, label, ls in [(df_train_sim, "train sim", "-"), (df_val_sim, "val sim (held-out)", "--")]:
        # mark trained-on snaps with filled markers
        filled   = df_s[df_s["trained_on"]]
        unfilled = df_s[~df_s["trained_on"]]

        for df_sub, marker in [(filled, "o"), (unfilled, "^")]:
            axes[0].plot(df_sub["a"], df_sub["test_R"],  ls=ls, marker=marker, lw=2,
                         label=f"{label} {'(trained)' if marker=='o' else '(interp.)'}")
            axes[1].plot(df_sub["a"], df_sub["test_MSE"], ls=ls, marker=marker, lw=2,
                         label=f"{label} {'(trained)' if marker=='o' else '(interp.)'}")

    axes[0].set_xlabel("a"); axes[0].set_ylabel("Test R (patch test region)")
    axes[0].set_title("Temporal generalisation: Test R vs a")
    axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

    axes[1].axhline(0, color="k", ls="--", lw=0.8)
    axes[1].set_xlabel("a"); axes[1].set_ylabel("MSE improvement %")
    axes[1].set_title("Force MSE improvement vs a")
    axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)

    plt.suptitle(f"{STAGE_LABEL} — temporal generalisation (●=trained epoch  ▲=interpolated)", fontsize=11)
    plt.tight_layout(); plt.show()

    # Summary
    print(f"\nMean test R across all snaps:")
    print(f"  Train sim: {df_train_sim['test_R'].mean():.4f}  (trained snaps: {df_train_sim[df_train_sim['trained_on']]['test_R'].mean():.4f}  other: {df_train_sim[~df_train_sim['trained_on']]['test_R'].mean():.4f})")
    print(f"  Val sim:   {df_val_sim['test_R'].mean():.4f}  (trained snaps: {df_val_sim[df_val_sim['trained_on']]['test_R'].mean():.4f}  other: {df_val_sim[~df_val_sim['trained_on']]['test_R'].mean():.4f})")

## Section 5 — Cross-simulation generalisation

Evaluates on ALL training sims and the held-out val sim at the same snapshot.
Shows whether the model trained on sims 0-4 generalises to sim 5.

In [ ]:
if MLP_PARAMS is not None:
    SNAP_MID = SNAPS_TRAIN[len(SNAPS_TRAIN) // 2]   # mid training snap
    rows_cs = []
    all_sims = SIM_IDS_TRAIN + SIM_IDS_VAL
    _fp_jit2 = jax.jit(partial(compute_force_pair, mesh_lr=MESH_LR, mesh_hr=MESH_HR))

    for sim_id in all_sims:
        p, v, a_s = load_single_snapshot(DATA_DIR, sim_id, SNAP_MID, N_PART, BOX_SIZE)
        p_lr = p * SCALE_TO_LR
        fc, ff, df = _fp_jit2(p_lr)
        feats, _ = snapshot_features(p, neighbor_idx, N_PART, USE_STRAIN, USE_INVS)
        vf = v if USE_VEL else jnp.zeros_like(v)
        mlp_p = np.asarray(jax.device_get(
            jax.jit(lag_model.apply)(MLP_PARAMS, feats, vf, jnp.array(float(a_s)))
        ))
        if CNN_MODEL is not None:
            v_lr = v * SCALE_TO_LR
            cnn_p = np.asarray(jax.device_get(jax.jit(lambda pp, vv, aa:
                compute_cnn_massres_correction(CNN_MODEL, CNN_PARAMS, pp, vv, aa, MESH_LR)
            )(p_lr, v_lr, jnp.array(float(a_s)))))
        else:
            cnn_p = np.zeros_like(mlp_p)

        total = cnn_p + mlp_p
        df_np = np.asarray(jax.device_get(df))
        fc_np = np.asarray(jax.device_get(fc))
        ff_np = np.asarray(jax.device_get(ff))

        full_idx = np.arange(len(total))
        r_full = float(np.mean([pearsonr(total[full_idx,c], df_np[full_idx,c])[0] for c in range(3)]))
        base = float(np.mean((fc_np - ff_np)**2))
        mse_i = (1 - float(np.mean((fc_np + total - ff_np)**2)) / base) * 100

        rows_cs.append(dict(
            sim=sim_id,
            in_train=sim_id in SIM_IDS_TRAIN,
            R_full=r_full, MSE_improv=mse_i,
        ))
        del p, v, fc, ff, df, feats, mlp_p, total

    df_cs = pd.DataFrame(rows_cs)
    print(f"Snap={SNAP_MID}  a≈{a_s:.3f}  —  cross-sim R\n")
    print(df_cs.to_string(index=False, float_format="{:.4f}".format))

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    colors = ["tomato" if r else "steelblue" for r in df_cs["in_train"]]
    axes[0].bar(df_cs["sim"].astype(str), df_cs["R_full"],   color=colors, alpha=0.8)
    axes[0].set_xlabel("sim id"); axes[0].set_ylabel("R (full sim)")
    axes[0].set_title("R per sim (red=trained, blue=val)")
    axes[0].set_ylim(0, 1)

    axes[1].bar(df_cs["sim"].astype(str), df_cs["MSE_improv"], color=colors, alpha=0.8)
    axes[1].axhline(0, color="k", lw=0.8)
    axes[1].set_xlabel("sim id"); axes[1].set_ylabel("MSE improvement %")
    axes[1].set_title("MSE improvement per sim")

    plt.suptitle(f"Cross-sim generalisation  ({STAGE_LABEL})  snap={SNAP_MID}", fontsize=11)
    plt.tight_layout(); plt.show()

## Section 6 — Spatial error maps

In [ ]:
if MLP_PARAMS is not None:
    # Reload inspect snapshot (computed in Section 3)
    err_all = np.sqrt(np.sum((total_np - df_np)**2, axis=-1))
    f_corrected = fc_np + total_np

    mse_base = float(np.mean((fc_np - ff_np)**2))
    mse_corr = float(np.mean((f_corrected - ff_np)**2))
    print(f"MSE improvement full sim: {(1 - mse_corr/mse_base)*100:.1f}%")
    print(f"  train region:           {(1 - np.mean((f_corrected[train_idx]-ff_np[train_idx])**2) / np.mean((fc_np[train_idx]-ff_np[train_idx])**2))*100:.1f}%")
    print(f"  test  region:           {(1 - np.mean((f_corrected[test_idx] -ff_np[test_idx])**2)  / np.mean((fc_np[test_idx] -ff_np[test_idx])**2))*100:.1f}%")

    slab_mask = np.abs(pos_np[:, 2] % N_PART - N_PART//2) < max(1, N_PART//20)
    px, py    = pos_np[slab_mask, 0], pos_np[slab_mask, 1]
    pred_mag  = np.sqrt(np.sum(total_np**2, axis=-1))

    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    for ax, (C, title, cmap) in zip(axes, [
        (df_mag[slab_mask],   "|ΔF| ground truth",           "Oranges"),
        (pred_mag[slab_mask], f"|ΔF pred| ({STAGE_LABEL})",  "Blues"),
        (err_all[slab_mask],  "|pred − target|",              "Reds"),
        (np.where(np.isin(np.where(slab_mask)[0], test_idx), 1.0, 0.0),
                              "Region (0=train, 1=test)",     "RdBu"),
    ]):
        hb = ax.hexbin(px, py, C=C, gridsize=55, cmap=cmap, reduce_C_function=np.mean)
        plt.colorbar(hb, ax=ax); ax.set_title(title)

    plt.suptitle(f"Spatial maps  a={a:.3f}  {STAGE_LABEL}", fontsize=11)
    plt.tight_layout(); plt.show()

## Section 7 — On-the-fly fine-tuning 🚀

**Concept**: given a new simulation snapshot (never seen during pre-training),
take a small Lagrangian patch (12.5% of particles), fine-tune the pre-trained
MLP for ~100 gradient steps, then apply to ALL particles.

**Goal**: R_after > R_before on the test region (generalised from patch).

This is the core of the on-the-fly fine-tuning pipeline. The sub-region is chosen
to be computationally cheap (≪ N³ particles) while providing enough diversity for
rapid adaptation.

In [ ]:
# ── On-the-fly config ────────────────────────────────────────────────────────
OTF_SIM_ID    = SIM_IDS_VAL[0]   # held-out sim (never seen during training)
OTF_SNAP_ID   = 5                 # snap to fine-tune on (a ≈ 0.6)
OTF_PATCH_N   = N_PART // 2       # fine-tune patch: (N/2)³ = 12.5% of particles
OTF_N_STEPS   = 100               # gradient steps (fast: ~5s on GPU)
OTF_LR        = 1e-4              # fine-tuning learning rate (smaller than training)
OTF_PATCH_SEED = 0                # different seed than training split

otf_frac = (OTF_PATCH_N / N_PART) ** 3
print(f"OTF setup:")
print(f"  sim={OTF_SIM_ID} (held-out)  snap={OTF_SNAP_ID}")
print(f"  fine-tune patch: {OTF_PATCH_N}³ = {OTF_PATCH_N**3:,} / {N_PART**3:,} particles ({otf_frac:.1%})")
print(f"  fine-tune steps: {OTF_N_STEPS}  lr={OTF_LR}")

In [ ]:
from train_subregion_forceres import make_patch_split

def finetune_on_patch(init_params, pos, vel, a_snap,
                      patch_n, patch_seed, n_steps, lr,
                      mesh_lr, mesh_hr, scale_to_lr, use_vel,
                      cnn_model=None, cnn_params=None):
    """Fine-tune MLP on a Lagrangian patch and return updated params."""
    ft_idx, _, _ = make_patch_split(N_PART, patch_n, seed=patch_seed)

    pos_lr_j = pos * scale_to_lr
    fc, ff, df = jax.jit(partial(compute_force_pair, mesh_lr=mesh_lr, mesh_hr=mesh_hr))(pos_lr_j)
    feats, _   = snapshot_features(pos, neighbor_idx, N_PART, USE_STRAIN, USE_INVS)
    vf         = vel if use_vel else jnp.zeros_like(vel)
    a_j        = jnp.array(float(a_snap))

    df_target = jnp.array(np.asarray(jax.device_get(df)))
    if cnn_model is not None:
        v_lr = vel * scale_to_lr
        cnn_p = jax.jit(lambda pp, vv, aa:
            compute_cnn_massres_correction(cnn_model, cnn_params, pp, vv, aa, mesh_lr)
        )(pos_lr_j, v_lr, a_j)
        df_target = df_target - cnn_p

    feats_patch  = feats[ft_idx]
    vf_patch     = vf[ft_idx]
    target_patch = df_target[ft_idx]

    optimizer = optax.adam(lr)
    opt_state = optimizer.init(init_params)
    params    = init_params

    @jax.jit
    def step(params, opt_state):
        def loss_fn(p):
            pred = lag_model.apply(p, feats_patch, vf_patch, a_j)
            return jnp.mean((pred - target_patch) ** 2)
        loss, grads = jax.value_and_grad(loss_fn)(params)
        updates, opt_state_new = optimizer.update(grads, opt_state)
        return optax.apply_updates(params, updates), opt_state_new, loss

    losses = []
    for i in range(n_steps):
        params, opt_state, loss = step(params, opt_state)
        if i % 20 == 0:
            losses.append((i, float(loss)))
            print(f"  step {i:3d}  loss={float(loss):.4e}")

    return params, losses, ft_idx


if MLP_PARAMS is not None:
    print(f"Loading OTF snapshot: sim={OTF_SIM_ID} snap={OTF_SNAP_ID}")
    pos_otf, vel_otf, a_otf = load_single_snapshot(
        DATA_DIR, OTF_SIM_ID, OTF_SNAP_ID, N_PART, BOX_SIZE
    )
    print(f"a={a_otf:.3f}")

    # ── Baseline: pre-trained model ───────────────────────────────────────────
    total_before, cnn_b, mlp_b, df_otf, fc_otf, ff_otf = predict_all(
        MLP_PARAMS, CNN_MODEL, CNN_PARAMS, pos_otf, vel_otf, a_otf,
        MESH_LR, MESH_HR, SCALE_TO_LR, USE_VEL
    )
    full_idx = np.arange(len(total_before))
    R_before = float(np.mean([pearsonr(total_before[full_idx, c], df_otf[full_idx, c])[0] for c in range(3)]))
    print(f"\nR_before (pre-trained): {R_before:.4f}")

    # ── Baseline: linear force interpolation ──────────────────────────────────
    # Evaluate pre-trained model at the two bracketing TRAINING a-values using
    # the SAME positions (pos_otf), then linearly interpolate in a.
    scale_factors_all = np.load(DATA_DIR / "scale_factors.npy")
    a_train_list = [(s, float(scale_factors_all[s])) for s in SNAPS_TRAIN]

    below_snaps = [(s, a) for s, a in a_train_list if a <= a_otf + 1e-6]
    above_snaps = [(s, a) for s, a in a_train_list if a >= a_otf - 1e-6]
    s_lo, a_lo = max(below_snaps, key=lambda x: x[1]) if below_snaps else a_train_list[0]
    s_hi, a_hi = min(above_snaps, key=lambda x: x[1]) if above_snaps else a_train_list[-1]

    print(f"\nLinear interp baseline: snap {s_lo} (a={a_lo:.3f}) ↔ snap {s_hi} (a={a_hi:.3f})")
    pred_lo, *_ = predict_all(MLP_PARAMS, CNN_MODEL, CNN_PARAMS, pos_otf, vel_otf, a_lo,
                               MESH_LR, MESH_HR, SCALE_TO_LR, USE_VEL)
    pred_hi, *_ = predict_all(MLP_PARAMS, CNN_MODEL, CNN_PARAMS, pos_otf, vel_otf, a_hi,
                               MESH_LR, MESH_HR, SCALE_TO_LR, USE_VEL)

    if abs(a_hi - a_lo) < 1e-8:
        t_interp  = 0.0
        df_linear = pred_lo
    else:
        t_interp  = (a_otf - a_lo) / (a_hi - a_lo)
        df_linear = (1.0 - t_interp) * pred_lo + t_interp * pred_hi

    R_linear = float(np.mean([pearsonr(df_linear[full_idx, c], df_otf[full_idx, c])[0] for c in range(3)]))
    print(f"R_linear_interp (t={t_interp:.2f}): {R_linear:.4f}")

    # ── Fine-tune MLP ─────────────────────────────────────────────────────────
    print(f"\nFine-tuning MLP on {OTF_PATCH_N}³ patch ({OTF_N_STEPS} steps)…")
    ft_params, ft_losses, ft_idx = finetune_on_patch(
        MLP_PARAMS, pos_otf, vel_otf, a_otf,
        OTF_PATCH_N, OTF_PATCH_SEED, OTF_N_STEPS, OTF_LR,
        MESH_LR, MESH_HR, SCALE_TO_LR, USE_VEL,
        CNN_MODEL, CNN_PARAMS
    )

    total_after, _, _, _, _, _ = predict_all(
        ft_params, CNN_MODEL, CNN_PARAMS, pos_otf, vel_otf, a_otf,
        MESH_LR, MESH_HR, SCALE_TO_LR, USE_VEL
    )
    R_after = float(np.mean([pearsonr(total_after[full_idx, c], df_otf[full_idx, c])[0] for c in range(3)]))

    print(f"\n{'═'*50}")
    print(f"  R_linear_interp  (no adapt):   {R_linear:.4f}")
    print(f"  R_before         (pre-train):   {R_before:.4f}   Δ vs interp: {R_before - R_linear:+.4f}")
    print(f"  R_after          (MLP OTF):     {R_after:.4f}   Δ vs interp: {R_after  - R_linear:+.4f}")
    print(f"{'═'*50}")

    for label, idx in [("fine-tune patch", ft_idx),
                       ("rest",            np.setdiff1d(full_idx, ft_idx))]:
        r_b = float(np.mean([pearsonr(total_before[idx, c], df_otf[idx, c])[0] for c in range(3)]))
        r_a = float(np.mean([pearsonr(total_after[idx, c],  df_otf[idx, c])[0] for c in range(3)]))
        r_l = float(np.mean([pearsonr(df_linear[idx, c],    df_otf[idx, c])[0] for c in range(3)]))
        print(f"  {label:<20}  R_linear={r_l:.4f}  R_before={r_b:.4f}  R_after={r_a:.4f}")
else:
    print("⚠  Load MLP checkpoint first (run train_pretrain_forceres.py)")

In [ ]:
if MLP_PARAMS is not None:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    # Loss curve
    steps_l, loss_l = zip(*ft_losses)
    axes[0].plot(steps_l, loss_l, "o-", lw=2, color="C1")
    axes[0].set_xlabel("step"); axes[0].set_ylabel("patch MSE")
    axes[0].set_title("Fine-tuning loss curve")
    axes[0].set_yscale("log"); axes[0].grid(alpha=0.3)

    # Before vs After scatter (full sim, z-component)
    ss = np.random.default_rng(99).choice(len(full_idx), min(40_000, len(full_idx)), replace=False)
    tgt = df_otf[ss, 2]
    lim = max(abs(np.percentile(tgt, 1)), abs(np.percentile(tgt, 99))) * 1.15

    axes[1].hexbin(tgt, total_before[ss, 2], gridsize=60, cmap="Blues", norm=LogNorm(),
                   mincnt=1, extent=[-lim, lim, -lim, lim])
    axes[1].plot([-lim, lim], [-lim, lim], "r--", lw=1)
    axes[1].set_title(f"Before  R={R_before:.4f}  (z component)")
    axes[1].set_xlabel("ΔF_z target"); axes[1].set_ylabel("predicted")

    axes[2].hexbin(tgt, total_after[ss, 2], gridsize=60, cmap="Greens", norm=LogNorm(),
                   mincnt=1, extent=[-lim, lim, -lim, lim])
    axes[2].plot([-lim, lim], [-lim, lim], "r--", lw=1)
    axes[2].set_title(f"After   R={R_after:.4f}  (z component)")
    axes[2].set_xlabel("ΔF_z target"); axes[2].set_ylabel("predicted")

    plt.suptitle(f"On-the-fly fine-tuning  sim={OTF_SIM_ID}  snap={OTF_SNAP_ID}  (a={a_otf:.3f})\n"
                 f"patch={OTF_PATCH_N}³ ({otf_frac:.1%})  {OTF_N_STEPS} steps  lr={OTF_LR}", fontsize=10)
    plt.tight_layout(); plt.show()

In [ ]:
# ── Sweep: fine-tune at every snapshot of the val sim ──────────────────────
# This shows whether on-the-fly adaptation helps consistently across time.
# (Takes a few minutes; skip if just doing a quick check.)

RUN_OTF_SWEEP = False   # set True to run

if RUN_OTF_SWEEP and MLP_PARAMS is not None:
    _sf_all   = np.load(DATA_DIR / "scale_factors.npy")
    _a_train  = [(s, float(_sf_all[s])) for s in SNAPS_TRAIN]

    rows_otf = []
    for snap_id in ALL_SNAPS:
        p_s, v_s, a_s = load_single_snapshot(DATA_DIR, OTF_SIM_ID, snap_id, N_PART, BOX_SIZE)
        a_s_f = float(a_s)

        # Baseline: pre-trained
        tot_b, _, _, df_s, fc_s, ff_s = predict_all(
            MLP_PARAMS, CNN_MODEL, CNN_PARAMS, p_s, v_s, a_s_f,
            MESH_LR, MESH_HR, SCALE_TO_LR, USE_VEL
        )
        full = np.arange(len(tot_b))
        r_b  = float(np.mean([pearsonr(tot_b[full, c], df_s[full, c])[0] for c in range(3)]))

        # Baseline: linear force interpolation
        bel = [(s, a) for s, a in _a_train if a <= a_s_f + 1e-6]
        abv = [(s, a) for s, a in _a_train if a >= a_s_f - 1e-6]
        s_lo, a_lo = max(bel, key=lambda x: x[1]) if bel else _a_train[0]
        s_hi, a_hi = min(abv, key=lambda x: x[1]) if abv else _a_train[-1]
        p_lo, *_ = predict_all(MLP_PARAMS, CNN_MODEL, CNN_PARAMS, p_s, v_s, a_lo,
                               MESH_LR, MESH_HR, SCALE_TO_LR, USE_VEL)
        p_hi, *_ = predict_all(MLP_PARAMS, CNN_MODEL, CNN_PARAMS, p_s, v_s, a_hi,
                               MESH_LR, MESH_HR, SCALE_TO_LR, USE_VEL)
        if abs(a_hi - a_lo) < 1e-8:
            df_lin = p_lo
        else:
            t = (a_s_f - a_lo) / (a_hi - a_lo)
            df_lin = (1.0 - t) * p_lo + t * p_hi
        r_l = float(np.mean([pearsonr(df_lin[full, c], df_s[full, c])[0] for c in range(3)]))

        # Fine-tune MLP
        ft_p, _, _ = finetune_on_patch(
            MLP_PARAMS, p_s, v_s, a_s_f,
            OTF_PATCH_N, OTF_PATCH_SEED, OTF_N_STEPS, OTF_LR,
            MESH_LR, MESH_HR, SCALE_TO_LR, USE_VEL, CNN_MODEL, CNN_PARAMS
        )
        tot_a, _, _, _, _, _ = predict_all(
            ft_p, CNN_MODEL, CNN_PARAMS, p_s, v_s, a_s_f,
            MESH_LR, MESH_HR, SCALE_TO_LR, USE_VEL
        )
        r_a = float(np.mean([pearsonr(tot_a[full, c], df_s[full, c])[0] for c in range(3)]))

        rows_otf.append(dict(
            snap=snap_id, a=a_s_f,
            R_linear=r_l, R_before=r_b, R_after=r_a,
            delta_vs_linear=r_a - r_l,
        ))
        print(f"snap {snap_id}  a={a_s_f:.3f}  R_linear={r_l:.4f}  R_before={r_b:.4f}  R_after={r_a:.4f}  Δ={r_a-r_l:+.4f}")
        del p_s, v_s

    df_otf_sweep = pd.DataFrame(rows_otf)

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    ax = axes[0]
    ax.plot(df_otf_sweep["a"], df_otf_sweep["R_linear"], "s--", c="gray",      lw=2, label="R_linear (interp)")
    ax.plot(df_otf_sweep["a"], df_otf_sweep["R_before"], "o--", c="steelblue", lw=2, label="R_before (pre-train)")
    ax.plot(df_otf_sweep["a"], df_otf_sweep["R_after"],  "o-",  c="tomato",    lw=2, label="R_after (MLP OTF)")
    ax.fill_between(df_otf_sweep["a"], df_otf_sweep["R_linear"], df_otf_sweep["R_after"],
                    alpha=0.15, color="green", label="gain vs interp")
    ax.set_xlabel("a"); ax.set_ylabel("R (full sim)")
    ax.set_title("R vs a — OTF fine-tune sweep")
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

    ax = axes[1]
    ax.bar(df_otf_sweep["a"].astype(str),
           df_otf_sweep["R_after"] - df_otf_sweep["R_linear"],
           color=["tomato" if v > 0 else "steelblue" for v in df_otf_sweep["R_after"] - df_otf_sweep["R_linear"]],
           alpha=0.8)
    ax.axhline(0, color="k", lw=0.8)
    ax.set_xlabel("a"); ax.set_ylabel("R_after − R_linear")
    ax.set_title("Net gain of MLP OTF vs linear interp baseline")
    ax.grid(alpha=0.3, axis="y")

    plt.suptitle(f"OTF sweep  sim={OTF_SIM_ID}  {OTF_N_STEPS} steps  patch={OTF_PATCH_N}³", fontsize=11)
    plt.tight_layout(); plt.show()

    print("\n" + df_otf_sweep.to_string(index=False, float_format="{:.4f}".format))

## Sección 7b — Fine-tuning on-the-fly sobre la CNN

El MLP no mejora (o empeora ligeramente) porque sus features lagrangianas no capturan
la estructura no-lineal específica del snapshot. La CNN opera en el espacio de la densidad
y tiene mucha más capacidad representacional.

**Estrategia**: supervisar la CNN directamente sobre el **potencial diferencial ΔΦ** (no fuerzas).

- `compute_cnn_massres_correction` computa fuerzas como `jax.grad(Σ CNN(params,...))(pos)`.
  Si supervisáramos fuerzas, necesitaríamos `d/d_params [d/d_pos Σ CNN]` → doble gradiente.
- Supervisando ΔΦ directamente, el loss es de primer orden y mucho más eficiente.
- Target: `ΔΦ_target_i = Φ_fine(pos_hr_i) − Φ_coarse(pos_lr_i)`
- Al inference time, las fuerzas se siguen computando vía `jax.grad` con los params actualizados.

In [ ]:
from jaxpm.pm import get_delta, potential_kgrid_to_force_at_pos
from jaxpm.kernels import fftk
from jaxpm.painting import cic_read

@partial(jax.jit, static_argnums=(1, 2, 3))
def compute_delta_phi(pos_j, mesh_lr, mesh_hr, scale_to_lr):
    """ΔΦ_i = Φ_fine(pos_hr_i) − Φ_coarse(pos_lr_i)  shape=[N].
    potential_kgrid_to_force_at_pos(..., return_potential=True) returns the potential
    GRID (mesh³), not per-particle values. We use cic_read to interpolate to particles."""
    pos_lr_j   = pos_j * scale_to_lr
    pos_hr_j   = pos_j * scale_to_lr * (mesh_hr / mesh_lr)
    pos_lr_mod = jnp.mod(pos_lr_j, mesh_lr)
    pos_hr_mod = jnp.mod(pos_hr_j, mesh_hr)

    # Coarse potential grid (128³) → CIC readout → per-particle [N]
    kvec_lr = fftk((mesh_lr,) * 3)
    d_lr    = get_delta(pos_lr_mod, (mesh_lr,) * 3)
    _, phi_lr_grid = potential_kgrid_to_force_at_pos(
        jnp.fft.rfftn(d_lr), pos_lr_mod, kvec_lr, return_potential=True)
    phi_lr = cic_read(phi_lr_grid, pos_lr_mod)   # [N]

    # Fine potential grid (256³) → CIC readout → per-particle [N]
    kvec_hr = fftk((mesh_hr,) * 3)
    d_hr    = get_delta(pos_hr_mod, (mesh_hr,) * 3)
    _, phi_hr_grid = potential_kgrid_to_force_at_pos(
        jnp.fft.rfftn(d_hr), pos_hr_mod, kvec_hr, return_potential=True)
    phi_hr = cic_read(phi_hr_grid, pos_hr_mod)   # [N]

    return phi_hr - phi_lr   # [N]  target ΔΦ per particle


if MLP_PARAMS is not None and CNN_MODEL is not None:
    pos_otf_j = jnp.array(np.asarray(jax.device_get(pos_otf)))
    vel_otf_j = jnp.array(np.asarray(jax.device_get(vel_otf)))

    print("Calculando ΔΦ target (Φ_fine − Φ_coarse)…")
    delta_phi_target = compute_delta_phi(pos_otf_j, MESH_LR, MESH_HR, SCALE_TO_LR)

    CNN_PATCH_SEED = 1   # semilla distinta al MLP → parche diferente = mejor test de generalización
    ft_cnn_idx, _, _ = make_patch_split(N_PART, OTF_PATCH_N, seed=CNN_PATCH_SEED)

    dphi_np = np.asarray(jax.device_get(delta_phi_target))
    print(f"ΔΦ_target   mean={dphi_np.mean():.4e}   std={dphi_np.std():.4e}")
    print(f"Parche CNN: {len(ft_cnn_idx):,} partículas (seed={CNN_PATCH_SEED})")
elif CNN_MODEL is None:
    print("⚠  CNN no cargado — saltando sección 7b.")

In [ ]:
CNN_OTF_STEPS = 150    # más pasos que MLP (la CNN tiene más params)
CNN_OTF_LR    = 5e-5   # conservador para no destruir el pre-entrenamiento

if MLP_PARAMS is not None and CNN_MODEL is not None:
    # ── Inputs fijos (sin grad a través de éstos) ─────────────────────────────
    pos_lr_cnn  = jax.lax.stop_gradient(jnp.mod(pos_otf_j * SCALE_TO_LR, MESH_LR))
    vel_lr_cnn  = jax.lax.stop_gradient(vel_otf_j * SCALE_TO_LR)
    a_j_cnn     = jnp.array(float(a_otf))

    kvec_cnn    = fftk((MESH_LR,) * 3)
    delta_cnn   = get_delta(pos_lr_cnn, (MESH_LR,) * 3)
    delta_k_cnn = jnp.fft.rfftn(delta_cnn)
    _, pm_pot_cnn = potential_kgrid_to_force_at_pos(
        delta_k_cnn, pos_lr_cnn, kvec_cnn, return_potential=True)
    grid_data_cnn = jax.lax.stop_gradient(jnp.stack([pm_pot_cnn, delta_cnn], axis=-1))

    target_cnn_patch = jax.lax.stop_gradient(delta_phi_target[ft_cnn_idx])
    pos_patch_cnn    = jax.lax.stop_gradient(pos_lr_cnn[ft_cnn_idx])
    vel_patch_cnn    = jax.lax.stop_gradient(vel_lr_cnn[ft_cnn_idx])

    # ── Fine-tuning loop ──────────────────────────────────────────────────────
    cnn_optimizer = optax.adam(CNN_OTF_LR)
    cnn_params_ft = CNN_PARAMS
    cnn_opt_state = cnn_optimizer.init(cnn_params_ft)

    @jax.jit
    def cnn_step(params, opt_state):
        def loss_fn(p):
            phi_pred = CNN_MODEL.apply(p, grid_data_cnn, pos_patch_cnn, a_j_cnn, vel_patch_cnn)[:, 0]
            return jnp.mean((phi_pred - target_cnn_patch) ** 2)
        loss, grads = jax.value_and_grad(loss_fn)(params)
        updates, new_state = cnn_optimizer.update(grads, opt_state)
        return optax.apply_updates(params, updates), new_state, loss

    print(f"Fine-tuning CNN (ΔΦ supervision)  {CNN_OTF_STEPS} pasos  lr={CNN_OTF_LR}…")
    cnn_losses = []
    for i in range(CNN_OTF_STEPS):
        cnn_params_ft, cnn_opt_state, loss = cnn_step(cnn_params_ft, cnn_opt_state)
        if i % 30 == 0:
            cnn_losses.append((i, float(loss)))
            print(f"  step {i:3d}  ΔΦ-loss={float(loss):.4e}")

    print("Fine-tuning CNN listo.")

In [ ]:
if MLP_PARAMS is not None and CNN_MODEL is not None:
    # ── Evalúa CNN pre-entrenada vs fine-tuneada (solo CNN, sin MLP) ──────────
    def cnn_only_correction(cnn_p, pos_j, vel_j, a_val):
        pos_lr = pos_j * SCALE_TO_LR
        vel_lr = vel_j * SCALE_TO_LR
        return np.asarray(jax.device_get(
            jax.jit(lambda p, pv, vv, av:
                compute_cnn_massres_correction(CNN_MODEL, p, pv, vv, av, MESH_LR)
            )(cnn_p, pos_lr, vel_lr, jnp.array(float(a_val)))
        ))

    cnn_corr_before = cnn_only_correction(CNN_PARAMS,    pos_otf_j, vel_otf_j, a_otf)
    cnn_corr_after  = cnn_only_correction(cnn_params_ft, pos_otf_j, vel_otf_j, a_otf)

    def r_full(pred):
        return float(np.mean([pearsonr(pred[full_idx, c], df_otf[full_idx, c])[0] for c in range(3)]))

    R_cnn_before = r_full(cnn_corr_before)
    R_cnn_after  = r_full(cnn_corr_after)

    print(f"\n{'═'*52}")
    print(f"  Baseline lineal           R = {R_linear:.4f}")
    print(f"  CNN pre-trained           R = {R_cnn_before:.4f}   Δ vs lineal: {R_cnn_before - R_linear:+.4f}")
    print(f"  CNN fine-tuned  (OTF)     R = {R_cnn_after:.4f}   Δ vs lineal: {R_cnn_after  - R_linear:+.4f}")
    print(f"  MLP fine-tuned  (OTF)     R = {R_after:.4f}   Δ vs lineal: {R_after       - R_linear:+.4f}")
    print(f"{'═'*52}")

    for label, idx in [("parche CNN (ft)", ft_cnn_idx),
                       ("resto",           np.setdiff1d(full_idx, ft_cnn_idx))]:
        r_b = float(np.mean([pearsonr(cnn_corr_before[idx, c], df_otf[idx, c])[0] for c in range(3)]))
        r_a = float(np.mean([pearsonr(cnn_corr_after[idx, c],  df_otf[idx, c])[0] for c in range(3)]))
        print(f"  {label:<22}  R_before={r_b:.4f}  R_after={r_a:.4f}  Δ={r_a - r_b:+.4f}")

    # ── Plots ──────────────────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    steps_c, loss_c = zip(*cnn_losses)
    axes[0].plot(steps_c, loss_c, "o-", lw=2, color="C2")
    axes[0].set_xlabel("step"); axes[0].set_ylabel("ΔΦ MSE (parche)")
    axes[0].set_title("CNN OTF loss (ΔΦ supervisión)")
    axes[0].set_yscale("log"); axes[0].grid(alpha=0.3)

    ss3 = np.random.default_rng(7).choice(len(full_idx), min(40_000, len(full_idx)), replace=False)
    tgt = df_otf[ss3, 2]
    lim = max(abs(np.percentile(tgt, 1)), abs(np.percentile(tgt, 99))) * 1.15

    for ax, pred, title, cmap in [
        (axes[1], cnn_corr_before[ss3, 2], f"CNN pre-trained   R={R_cnn_before:.4f}", "Purples"),
        (axes[2], cnn_corr_after[ss3, 2],  f"CNN fine-tuned    R={R_cnn_after:.4f}",  "Greens"),
    ]:
        ax.hexbin(tgt, pred, gridsize=60, cmap=cmap, norm=LogNorm(),
                  mincnt=1, extent=[-lim, lim, -lim, lim])
        ax.plot([-lim, lim], [-lim, lim], "r--", lw=1)
        ax.set_title(title)
        ax.set_xlabel("ΔF_z target"); ax.set_ylabel("predicted")

    plt.suptitle(
        f"CNN on-the-fly fine-tuning  sim={OTF_SIM_ID}  a={a_otf:.3f}\n"
        f"parche={OTF_PATCH_N}³ (seed={CNN_PATCH_SEED})  {CNN_OTF_STEPS} pasos  lr={CNN_OTF_LR}  ΔΦ supervisión",
        fontsize=10
    )
    plt.tight_layout(); plt.show()